# Simple Plume Tracking

!jupyter nbconvert --to script 01-Generate_Tobac_Tracking_from_3D_output.ipynb

In [1]:
import sys
is_notebook = True if 'ipykernel' in sys.argv[0] else False

# ============================================================================
# PATHS & CONFIG
# ============================================================================
tracking_comp = False
debug, debug_step = False, 15
cell_extraction_type = ['integrated', 'extreme', 'vertical'] #'integrated', 'extreme', 
# threshold = 1e-10
max_altitude = 1500 # m
model_data_root  = '/work/bb1262/user/schimmel/cosmo-specs-torch/cosmo-specs-runs/'
#default
cs_runs = [
#     ["cs-eriswil__20251129_230943", "50x40", 0, 0, 1], 
#     ["cs-eriswil__20251129_230943", "50x40", 1, 0, 1], 
#     ["cs-eriswil__20251125_114053", "50x40", 1, 0, 1],    #C,  ccn400
# #    ["cs-eriswil__20251209_001346", "200x160", 0, 0, 1], # 200x160
# #    ["cs-eriswil__20251209_001346", "200x160", 1, 0, 1], # 200x160
#     ["cs-eriswil__20251129_230943", "50x40", 0, 0, 1], # 50x40
#     ["cs-eriswil__20251129_230943", "50x40", 1, 0, 1], # 50x40
#     ["cs-eriswil__20251125_114053", "50x40", 0, 0, 1], # 50x40)] 
    # ["cs-eriswil__20251125_114053", "50x40", 1, 0, 1], # 50x40)] 
    ["cs-eriswil__20260123_180947", "200x160", 0, 0, 1],
    ["cs-eriswil__20260123_180947", "200x160", 1, 0, 1],
    ["cs-eriswil__20260123_180947", "200x160", 2, 0, 1],
    # ["cs-eriswil__20260121_131528", "50x40", 0, 0, 1],    #S=1, ccn400
    # ["cs-eriswil__20260127_211338", "50x40", 0, 1, 1],    #A=2, ccn400
    # ["cs-eriswil__20260127_211338", "50x40", 1, 1, 1],    #P=3, ccn400
    # ["cs-eriswil__20260121_131528", "50x40", 1, 0, 1],    #C=4, ccn400
    # #
    # ["cs-eriswil__20260210_113944", "50x40", 0, 0, 1],
    # ["cs-eriswil__20260210_113944", "50x40", 1, 1, 1],
    # ["cs-eriswil__20260211_194236", "50x40", 0, 1, 1],
    # ["cs-eriswil__20260211_194236", "50x40", 1, 0, 1],
    # ["cs-eriswil__20260211_194236", "50x40", 1, 0, 1],


  ]

if not is_notebook and len(sys.argv) > 1:
    idx_cs_run = int(sys.argv[1])
    cs_runs = [cs_runs[idx_cs_run]]



# feature selection, tracking (linking), and segmentation

parameter_features = {}
parameter_features["position_threshold"] = "extreme"  # weighted_diff
parameter_features["sigma_threshold"] = 1.
parameter_features["n_min_threshold"] = 0
parameter_features["target"] = "maximum"
parameter_features["vertical_coord"] = "altitude"
# parameter_features["time_cell_min"] = 60 # Minimum length in time that a cell must be tracked for to be considered a valid cell in seconds. Default is None.

In [2]:

from IPython.display import HTML, display
from tqdm.auto import tqdm
import sys, os, glob
import pylab as plt
import json
import pandas as pd
import seaborn as sns
from pathlib import Path

import traceback


import numpy as np
import xarray as xr
xr.set_options(keep_attrs=True)

import dask
from dask.diagnostics.progress import ProgressBar
import matplotlib.pyplot as plt
import matplotlib.colors as colors

from utilities import fetch_3d_data, convert_units_3d,  set_name_tick_params
from utilities import update_dataset_metadata, plot_3d_col_wrap, new_fjet2
from utilities import define_bin_boundaries, make_3d_preprocessor


import tobac
import iris
iris.FUTURE.date_microseconds = True



################################################################################
# prepare tobac input field, data needs to be converted to mass/number concentrations
def prep_tobac_input(ds_flare, ds_ref, standard_name=None, mask_threshold=1e-12):
    tobac_input = ds_flare - ds_ref
    tobac_input.attrs["units"] = "1/dm3"
    if "standard_name" in tobac_input.attrs and standard_name is None:
        tobac_input.attrs.pop("standard_name")
    else:
        tobac_input.attrs["standard_name"] = standard_name
    return xr.where(tobac_input < mask_threshold, mask_threshold, tobac_input)






## 2.X Plume Tracking and Extraction of Size-Resolved Microphysical Properties

Plume structures were identified and tracked in 5-D COSMO-SPECS output using the Tracking and Object-Based Analysis of Clouds (Tobac) toolkit (Heikenfeld et al., 2019). The tracking field was constructed as the difference of the size-integrated ice number concentrations ($n_f$) between flare and reference simulations, i.e. $n_i^{\mathrm{feature}} = n_f^{\mathrm{flare}} -  n_f^{\mathrm{ref}}$. The analysis domain was restricted to a flare-centred region extending two grid cells beyond the flare location and to altitudes above 1500\,m a.m.s.l., to exclude any non-target features from the cloud tracking output. The detection threshold was set such that values of $n_i^{\mathrm{feature}}\geq1\,L^{-1}$ are considered for tracking, see Omanovice et al. 2024. 

Feature detection was performed using Tobac's multi-threshold algorithm (`position_threshold="extreme"`, `target="maximum"`) with thresholds spanning several orders of magnitude. Detected features were linked into trajectories with a maximum permitted displacement of 100\,m\,s$^{-1}$ and a minimum cell lifetime of 60\,s. The Tobac segmentation module assigned grid cells to tracked features, producing a 4-D mask identifying the plume voxels at each time step.

Size-resolved microphysical properties were extracted along the tracked plume paths. The COSMO-SPECS bin-microphysics scheme resolves 66 logarithmically spaced size and mass bins spanning 1$\,$µm to several mm in diameter. At each time step, the 5-D spectral fields of frozen mass mixing ratio ($q_{\mathrm{fw}}$), liquid mass mixing ratio ($q_{\mathrm{w}}$), ice crystal number ($n_{\mathrm{f}}$), and cloud droplet number ($n_{\mathrm{w}}$) were masked to retain only plume voxels. Three extraction modes were applied:

- **Integrated:** For each time step $t$ and diameter bin $d$, the spectral variable $\psi(t, d)$ is computed as the mean over all plume voxels:
  $$\psi(t, d) = \frac{\sum_{i,j,k} \psi_{i,j,k}(t, d)}{N_{>0}(t, d)}$$
  where $i, j, k$ index altitude, latitude, and longitude, and $N_{>0}$ is the count of voxels with $\psi > 0$. This yields a 2-D array (time $\times$ diameter) per tracked cell.
- **Extreme:** For each time step $t$, values are sampled at the tracked extreme-value coordinates $(x_t, y_t, z_t)$, where the extreme corresponds to the local maximum identified by Tobac: $\psi(t, d) = \psi(x_t, y_t, z_t, d)$. This yields a 2-D array (time $\times$ diameter) per tracked cell.
- **Vertical:** Values are sampled at the horizontal extreme coordinates $(x_t, y_t)$ while retaining all altitude levels: $\psi(t, z, d) = \psi(x_t, y_t, z, d)$. This produces a 5-D array (time $\times$ altitude $\times$ diameter).

The resulting datasets were stored as NetCDF files with dimensions (time, diameter) or (time, diameter, altitude). The time–diameter plots in subsequent figures display the *integrated* fields, showing the evolution of the size distribution along the plume trajectory.


In [3]:

for cs_run, dom_xy, cs_run_idx, cs_run_idx_ref, threshold in cs_runs:
    #  ---  START OF THE MAIN EXECUTION ---  
    model_data_path = model_data_root + f'/RUN_ERISWILL_{dom_xy}x100/ensemble_output/{cs_run}/'
    extpar_file     = model_data_root + f'/RUN_ERISWILL_{dom_xy}x100/COS_in/extPar_Eriswil_{dom_xy}.nc'
    json_file       = glob.glob(model_data_path + "*.json")
    filelist_3d_nc     = sorted(glob.glob(model_data_path + f'3D_??????????????.nc'))

    print("\n[1] INPUT_ORG namelist file")
    with open(glob.glob(model_data_path + "*.json")[0], "r") as jsonfile: 
        meta = json.load(jsonfile)

    exp_names         = [f.split('/')[-1].split('_')[-1].split('.')[0] for f in filelist_3d_nc]
    lflare_runs       = [meta[exp]['INPUT_ORG']['sbm_par']['lflare'] for exp in exp_names]
    flare_exp_names   = [exp for exp in exp_names if meta[exp]['INPUT_ORG']['sbm_par']['lflare']]
    ref_exp_names     = [exp for exp in exp_names if not meta[exp]['INPUT_ORG']['sbm_par']['lflare']]
    
    if len(flare_exp_names) > 0:
        flare_exp_name    = flare_exp_names[cs_run_idx]
    else:
        raise ValueError(f"No flare experiments found for {cs_run}. Cannot proceed.")
    
    if len(ref_exp_names) > 0:
        ref_exp_name  = ref_exp_names[cs_run_idx_ref]
    else:
        raise ValueError(f"No reference experiments found for {cs_run}. Cannot proceed.")
        
    # try:    
    flare_exp_nc_file = [f for f in filelist_3d_nc if flare_exp_name in f][0]
    ref_exp_nc_file   = [f for f in filelist_3d_nc if ref_exp_name   in f][0]
    # except:
        # print(f"No reference experiments found for {cs_run}. Cannot proceed.")
        # continue

    print('-------------------------' , '-------------------------' )
    print('  all       flare runs :' , flare_exp_names )
    print('  all   reference runs :' , ref_exp_names )
    
    print('      flare / ref run             = ', flare_exp_name, ref_exp_name)
    print('      flare / ref emission rate   = ', meta[flare_exp_name]['INPUT_ORG']['flare_sbm']['flare_emission'], meta[ref_exp_name]['INPUT_ORG']['flare_sbm']['flare_emission'])
    print('      flare / ref ice shape param = ', meta[flare_exp_name]['INPUT_ORG']['sbm_par']['ishape'], meta[ref_exp_name]['INPUT_ORG']['sbm_par']['ishape'])
    
    if not tracking_comp:
        print('skipping tracking computation for ', flare_exp_name)
        continue
    

    attrs = {
        'flare_exp_name': flare_exp_name,
        'ref_exp_name': ref_exp_name,
        'flare_exp_nc_file': flare_exp_nc_file,
        'ref_exp_nc_file': ref_exp_nc_file,
        'resolution': '400m' if '50x40' in meta[flare_exp_name]['domain'] else '100m',
        'resolution_deg': 0.004 if '50x40' in meta[flare_exp_name]['domain'] else 0.001,
        'ydate_ini': str(meta[flare_exp_name]['INPUT_ORG']['runctl']['ydate_ini']),
        'hstart': str(meta[flare_exp_name]['INPUT_ORG']['runctl']['hstart']),
        'hstop': str(meta[flare_exp_name]['INPUT_ORG']['runctl']['hstop']),
        'dpi': 300,
        'pixel_size': (1920, 1080),
        'png_path': './',
        'flare_lat': 47.07425,
        'flare_lon': 7.90522,
        'flare_alt_idx': -meta[flare_exp_name]['INPUT_ORG']['flare_sbm']['flare_hight'],
        'lflare': meta[flare_exp_name]['INPUT_ORG']['sbm_par']['lflare'],
        'flare_emissions': meta[flare_exp_name]['INPUT_ORG']['flare_sbm']['flare_emission'],
        'flare_dn': meta[flare_exp_name]['INPUT_ORG']['flare_sbm']['flare_dn'],
        'ishape': meta[flare_exp_name]['INPUT_ORG']['sbm_par']['ishape'],
        'iimfr': meta[flare_exp_name]['INPUT_ORG']['sbm_par']['iimfr'],
        'origin_lat': 47.070522,
        'origin_lon': 7.872991,
        
    }
    reduced_domain = {'latitude':  slice(None, attrs['flare_lat'] + 2.*attrs['resolution_deg']), 
                      'longitude': slice(None, attrs['flare_lon'] + 2.*attrs['resolution_deg']),
                      'altitude':  slice(max_altitude, None)}

    print("\n[2] Loading 3D data...")
    # ds_3d_flare = fetch_3d_data(flare_exp_nc_file, extpar_file, meta[flare_exp_name]['INPUT_ORG'], var_sets=['meteo', 'spec'], chunks={'time': 4}) 
    preprocess = make_3d_preprocessor(flare_exp_nc_file, extpar_file, meta[flare_exp_name]['INPUT_ORG'])
    ds_3d_flare = xr.open_mfdataset(flare_exp_nc_file, preprocess=preprocess,chunks={'time': 4}, parallel=True)
    # File-level metadata (not available inside preprocess)
    run_id = flare_exp_nc_file.split('/')[-1].split('_')[1].split('.')[0]
    ds_3d_flare.attrs['ncfile'] = flare_exp_nc_file
    ds_3d_flare.attrs['run_id'] = run_id
    ds_3d_flare = update_dataset_metadata(ds_3d_flare)
    attrs['flare_alt'] = ds_3d_flare.altitude.values[attrs['flare_alt_idx']]
    ds_3d_flare  = ds_3d_flare.sel(reduced_domain)
    ds_3d_flare  = convert_units_3d(ds_3d_flare, ds_3d_flare["rho"])
    ds_3d_flare  = ds_3d_flare[['t', 'nf']]
    

    delta_x = 1e3 * np.mean(np.diff(ds_3d_flare.longitude.values)) * 111.13295254925466  # 1 degree of longitude in km
    delta_y = 1e3 * np.mean(np.diff(ds_3d_flare.latitude.values)) * 111.13295254925466  # 1 degree of latitude in km
    delta_t = np.mean(np.diff(ds_3d_flare.time.astype('datetime64[s]')).astype(float))
    print(f'debug:  delta_x: {delta_x:.3f} m    with n_x = {ds_3d_flare.longitude.size}')
    print(f'debug:  delta_y: {delta_y:.3f} m    with n_y = {ds_3d_flare.latitude.size}')
    print(f'debug:  delta_z: ---variable-- m    with n_z = {ds_3d_flare.altitude.size}')
    print(f'debug:  delta_t: {delta_t:.3f} s    with n_x = {ds_3d_flare.time.size}')
    

    print("\n[3] Loading 3D reference data...")
    # ds_3d_ref = fetch_3d_data(ref_exp_nc_file, extpar_file, meta[ref_exp_name]['INPUT_ORG'], var_sets=['meteo', 'spec'], chunks={'time': 4})
    preprocess = make_3d_preprocessor(ref_exp_nc_file, extpar_file, meta[ref_exp_name]['INPUT_ORG'])
    ds_3d_ref = xr.open_mfdataset(ref_exp_nc_file, preprocess=preprocess, chunks={'time': 4}, parallel=True)
    run_id = ref_exp_nc_file.split('/')[-1].split('_')[1].split('.')[0]
    ds_3d_ref.attrs['ncfile'] = ref_exp_nc_file
    ds_3d_ref.attrs['run_id'] = run_id
    ds_3d_ref = update_dataset_metadata(ds_3d_ref)
    ds_3d_ref  = ds_3d_ref.sel(reduced_domain)
    ds_3d_ref  = convert_units_3d(ds_3d_ref, ds_3d_ref["rho"])
    ds_3d_ref  = ds_3d_ref[['t', 'nf']]



    print("\n[4] Tobac cloud tracking...")
    
    parameter_features["threshold"] = [threshold, threshold*1e1, threshold*1e2]

    for var, spec_bounds in zip(['qi', 'qs'], [[30, 50], [50, None]]):
        
        qice_flare  = ds_3d_flare['nf'].isel(diameter=slice(*spec_bounds)).sum('diameter')
        qice_ref    = ds_3d_ref['nf'].isel(diameter=slice(*spec_bounds)).sum('diameter')
        tobac_input = prep_tobac_input(qice_flare, qice_ref, mask_threshold=1e-12)
        
        with ProgressBar() as pbar:
            tobac_input = tobac_input.persist()
        
        # Calculate spatial and temporal resolution
        delta_x = 1e3 * np.mean(np.diff(tobac_input.longitude.values)) * 111.13295254925466  # 1 degree of longitude in km
        delta_y = 1e3 * np.mean(np.diff(tobac_input.latitude.values)) * 111.13295254925466  # 1 degree of latitude in km
        delta_t = np.mean(np.diff(tobac_input.time.astype('datetime64[s]')).astype(float))
        print(f'debug:  delta_x: {delta_x:.3f} m    with n_x = {tobac_input.longitude.size}')
        print(f'debug:  delta_y: {delta_y:.3f} m    with n_y = {tobac_input.latitude.size}')
        print(f'debug:  delta_z: ---variable-- m    with n_z = {tobac_input.altitude.size}')
        print(f'debug:  delta_t: {delta_t:.3f} s    with n_x = {tobac_input.time.size}')

        dxy, dt = tobac.get_spacings(tobac_input, grid_spacing=np.max([delta_x, delta_y]), time_spacing=delta_t)
        
        # Setup output file names
        features_file      =  f'{model_data_path}/{flare_exp_name}_{var}_tobac_features.csv'
        tracks_file        =  f'{model_data_path}/{flare_exp_name}_{var}_tobac_track.csv'
        features_mask_file =  f'{model_data_path}/{flare_exp_name}_{var}_tobac_features_mask.csv'
        segm_mask_file     =  f'{model_data_path}/{flare_exp_name}_{var}_tobac_mask_xarray.nc'
        
        # print(f'debug:  min/max features      g/L    {tobac_input.min().values:.3e} to {tobac_input.max().values:.3e}')
        print(f'debug:  dxy: {dxy:.3f}')
        print(f'debug:  dt:  {dt:.3f}')
        
        
        # Tobac feature detection
        tobac_input_iris = tobac_input.to_iris()
        print(f'debug:  tobac_input_iris: {tobac_input_iris}\n')
        print('feature detection params:')
        print(parameter_features)
        
        # Start Feature Detection
        features = tobac.feature_detection_multithreshold( tobac_input_iris, dxy, **parameter_features )
        if features is not None and len(features) > 0:
            features.to_csv(features_file)
            print("\n[5] Tobac features written to: ", features_file)
        else:
            print(f"Warning [5]: Feature detection returned no data for {var}")
            print(f"    \n continue with next variable")
            continue
            
        # Link features to trajectories
        tracks = tobac.linking_trackpy( features, tobac_input_iris, vertical_coord=parameter_features["vertical_coord"], dt=dt, dxy=dxy, v_max=100,)
        tracks.to_csv(tracks_file)
        print("\n[6] Tobac tracks written to: ", tracks_file)
        
        # Segmentation
        iris_mask, features_mask = tobac.segmentation.segmentation( features, tobac_input_iris, dxy, threshold=parameter_features["threshold"][0], vertical_coord=parameter_features["vertical_coord"] )
        
        # save feature mask 
        features_mask.to_csv(features_mask_file)
        print("\n[7] Tobac features_mask written to: ", features_mask_file)
        
        # save segmentation mask
        xr_mask = xr.DataArray.from_iris(iris_mask)
        if os.path.exists(segm_mask_file):
            print(f'    Removed {segm_mask_file}')
            os.remove(segm_mask_file)
        xr_mask.to_netcdf(segm_mask_file, mode='w')
        print("\n[8] Tobac segm_mask written to: ", segm_mask_file)
        
        
        print(f'\n[9] Plot Facettes for var = {var} and tobac tracking:')
        print(f'features_file = {features_file}')
        print(f'tracks_file   = {tracks_file}')
        print(f'features_mask_file = {features_mask_file}')
        print(f'segm_mask_file     = {segm_mask_file}')
        
        if len(getattr(glob, 'glob', lambda x: [])(features_file)) == 0:
            print(f'No features file found for {features_file}')
            continue
        
        features = pd.read_csv(features_file).to_xarray()
        tracks   = pd.read_csv(tracks_file).to_xarray()
        features_mask = pd.read_csv(features_mask_file).to_xarray()
        categorie_dataset = xr.open_dataset(segm_mask_file)
        
        track_number = tracks['cell'].data
        cell_labels = tracks['feature'].data
        
        labels = categorie_dataset['segmentation_mask'].data
        for index, clabel in enumerate( cell_labels ):
            m = labels == clabel
            labels[m] = track_number[index]
            
        catmask = xr.zeros_like( categorie_dataset['segmentation_mask'] )
        catmask[:] = labels
        cells = np.unique( track_number )


        print(f'\n[10] Plot: Number of gridboxes in plume tracks: {len(cells)}')
        fig1, axs1 = plt.subplots(figsize=(10,3))
        for icell in cells:
            mask = catmask == icell
            mask = mask.sum(('longitude', 'latitude'))
            mask = xr.where(mask<1, np.nan, mask)
            pmmask = mask.plot(ax = axs1, x = 'time', add_colorbar=False, vmin=1, vmax=150)

        cbar = fig1.colorbar(pmmask )
        axs1.set_title(f'Size of the {var}')
        axs1.set_ylim((800, 1400))
        axs1.set_ylabel('# of gridboxes')
        fig1.savefig(f'./QL-01-Generate_Tobac_Tracking_from_3D_output/{flare_exp_name}_{var}_tobac_mask_plume_voxel.png', dpi=300)
        
        print(f'\n[11] tobac_input.sizes = {tobac_input.sizes}')
        tobac_tsize = tobac_input.time.size
        try:
            ds_plot = tobac_input.isel(time=slice(None, None, tobac_tsize//16 ))
            fig2, axs2 = plot_3d_col_wrap( ds_plot.sum('altitude'), figsize=(14,14), 
                                            cmap=new_fjet2, norm=colors.LogNorm(threshold, threshold*1e3), col_wrap=4)
            for icell in cells:
                track = xr.where(tracks['cell'] == icell, tracks, np.nan)
                for iax in axs2.flatten():
                    set_name_tick_params(iax)
                    iax.scatter( track['longitude'] , track['latitude'], s=40, marker="x", alpha = 0.9, label=f'track{icell}')
            
            axs2.flat[0].legend(loc='lower right')
            fig2.suptitle(f'{var} tracking')
            fig2.savefig(f'./QL-01-Generate_Tobac_Tracking_from_3D_output/{flare_exp_name}_{var}_tobac_tracking_top_view.png', dpi=300)
        except Exception as e:
            print(f'Error plotting {var} tracking')
            print(f'Exception: {type(e).__name__}: {e}')
            traceback.print_exc()
            

        



[1] INPUT_ORG namelist file
------------------------- -------------------------
  all       flare runs : ['20260123181143', '20260123181336', '20260123181750']
  all   reference runs : ['20260123180948']
      flare / ref run             =  20260123181143 20260123180948
      flare / ref emission rate   =  1000000.0 0.0
      flare / ref ice shape param =  1 1
skipping tracking computation for  20260123181143

[1] INPUT_ORG namelist file
------------------------- -------------------------
  all       flare runs : ['20260123181143', '20260123181336', '20260123181750']
  all   reference runs : ['20260123180948']
      flare / ref run             =  20260123181336 20260123180948
      flare / ref emission rate   =  1000000.0 0.0
      flare / ref ice shape param =  4 1
skipping tracking computation for  20260123181336

[1] INPUT_ORG namelist file
------------------------- -------------------------
  all       flare runs : ['20260123181143', '20260123181336', '20260123181750']
  all   ref

# 1.3 Extract Plume Path from tobac segmentation

In [4]:
# from utilities import extract_segmented_tracks_fast
def extract_segmented_tracks(ds, tracks, segm_mask, pre_track_minutes=5, delta_t=10, type='integrated'):
    """Vectorized version with pre-tracking timesteps."""
    cells = {}
    spec_var = ['qfw', 'qw', 'nf', 'nw']
    if type in ['integrated']:
        # set values outside the plume to 0
        ds = xr.where( segm_mask > 0.0, ds, 0.0 )
        
    for cell_id in np.unique(tracks['cell'].values):
        track = tracks.where(tracks['cell'] == cell_id, drop=True)
        
        # Add pre-tracking timesteps
        t0 = track['time'].astype('datetime64[ns]').values[0]
        t0_5min_earlier = t0 - np.timedelta64(pre_track_minutes, 'm')
        
        pre_times = pd.date_range(t0_5min_earlier, t0-np.timedelta64(10, 's'), freq='10s')
        n_pre_times = len(pre_times)

        pre_times = np.concatenate([pre_times, track['time'].values.astype('datetime64[ns]')])
        pre_lat   = np.concatenate([[ds.attrs['flare_lat']] * n_pre_times, track['latitude'].values])
        pre_lon   = np.concatenate([[ds.attrs['flare_lon']] * n_pre_times, track['longitude'].values])
        pre_alt   = np.concatenate([[ds.attrs['flare_alt']] * n_pre_times, track['altitude'].values])

        if type == 'extreme':
            ds_subset = ds.sel({'time': xr.DataArray(pre_times.astype('datetime64[ns]'), dims="path"),
                                'altitude': xr.DataArray(pre_alt, dims="path"),
                                'latitude': xr.DataArray(pre_lat, dims="path"),
                                'longitude': xr.DataArray(pre_lon, dims="path") }, method='nearest')
            temperature = ds_subset.t
            # delta_z = ds_subset.dz.values[..., np.newaxis]
            # cell_vol = ds.attrs['delta_x'] * ds.attrs['delta_y'] * delta_z
            cell_ds = ds_subset#[spec_var] #* cell_vol * 1e-3
            cell_ds['temperature'] = xr.DataArray(temperature, dims="path")
            # cell_ds['gridcell_volume'] = xr.DataArray(cell_vol, dims="path")
            cell_ds.attrs['kind'] = 'extreme'
            
        elif type == 'vertical':
            ds_subset = ds.sel({'time': xr.DataArray(pre_times.astype('datetime64[ns]'), dims="path"),
                                'latitude': xr.DataArray(pre_lat, dims="path"),
                                'longitude': xr.DataArray(pre_lon, dims="path") }, method='nearest')
            temperature = ds_subset.t
            # cell_vol = ds.attrs['delta_x'] * ds.attrs['delta_y']
            cell_ds = ds_subset#[spec_var] #* cell_vol * 1e-3
            cell_ds['temperature'] = xr.DataArray(temperature, dims=['path', 'altitude'])
            # cell_ds['gridcell_volume'] = xr.DataArray(cell_vol, dims="path")
            cell_ds.attrs['kind'] = 'vertical'
            
        elif type == 'integrated':
            ds_subset = ds.sel({ 'time': xr.DataArray(pre_times.astype('datetime64[ns]'), dims="path")}, method='nearest')
            temperature = ds_subset.t
            # delta_z = ds_subset.dz.values[..., np.newaxis]
            # cell_vol = ds.attrs['delta_x'] * ds.attrs['delta_y'] * delta_z  # pyright: ignore[reportUnusedVariable]
            cell_ds = ds_subset#[spec_var] #* cell_vol * 1e-3
            cell_ds = cell_ds.sum(['altitude', 'latitude', 'longitude'])/(segm_mask>0).sum(['altitude', 'latitude', 'longitude'])
            cell_ds['temperature'] = xr.DataArray(temperature.mean(['altitude', 'latitude', 'longitude']), dims="path")
            # cell_ds['gridcell_volume'] = xr.DataArray(cell_vol, dims="path")
            cell_ds.attrs['kind'] = 'integrated'
            
        else:
            raise ValueError(f"Invalid type: {type}")

        cells[cell_id] = cell_ds
    
    return cells



In [1]:

for cs_run, ep_dom, cs_run_idx, _, threshold in cs_runs:
    
    model_data_path = model_data_root + f'/RUN_ERISWILL_{ep_dom}x100/ensemble_output/{cs_run}/'
    extpar_file     = model_data_root + f'/RUN_ERISWILL_{ep_dom}x100/COS_in/extPar_Eriswil_{ep_dom}.nc'
    json_file       = glob.glob(model_data_path + "*.json")

    flist_3d = sorted(glob.glob(model_data_path + '3D_??????????????.nc'))
    exp_names = [f.split('/')[-1].split('_')[-1].split('.')[0] for f in flist_3d]
    with open(json_file[0], "r") as jsonfile:
        meta = json.load(jsonfile)

    flare_exp_name = [exp for exp in exp_names if meta[exp]['INPUT_ORG']['sbm_par']['lflare']][cs_run_idx]
    flare_exp_nc_file = [f for f in flist_3d if flare_exp_name in f][0]

    attrs = {
        'resolution': '400m' if '50x40' in meta[flare_exp_name]['domain'] else '100m',
        'resolution_deg': 0.004 if '50x40' in meta[flare_exp_name]['domain'] else 0.001,
        'flare_lat': 47.07425,
        'flare_lon': 7.90522,
        'flare_alt_idx': -meta[flare_exp_name]['INPUT_ORG']['flare_sbm']['flare_hight'],
    }
            
    print('model_data_path  =  ', model_data_path)
    print('    extpar_file  =  ', extpar_file)
    print('      json_file  =  ', json_file[0])
    print(' flare_exp_name  =  ', flare_exp_name)

    
    # Load and prepare 3D data
    reduced_domain = {'latitude':  slice(None, attrs['flare_lat'] + 2.*attrs['resolution_deg']), 
                      'longitude': slice(None, attrs['flare_lon'] + 2.*attrs['resolution_deg']),
                      'altitude':  slice(max_altitude, None)}
    
    # ds_3d = fetch_3d_data(flare_exp_nc_file, extpar_file, meta[flare_exp_name]['INPUT_ORG'], var_sets=['meteo', 'spec'], chunks={'time': 1, 'altitude':1})
    preprocess = make_3d_preprocessor(flare_exp_nc_file, extpar_file, meta[flare_exp_name]['INPUT_ORG'])
    ds = xr.open_mfdataset(flare_exp_nc_file, preprocess=preprocess,chunks={'time': 4}, parallel=True)
    
    # File-level metadata (not available inside preprocess)
    # update dataset attributes
    ds = ds.sel(reduced_domain)
    ds = update_dataset_metadata(ds)
    
    run_id = flare_exp_nc_file.split('/')[-1].split('_')[1].split('.')[0]
    ds.attrs.update(attrs)
    ds.attrs['flare_alt'] = ds.altitude.values[100-attrs['flare_alt_idx']]
    ds.attrs['run_id']    = run_id
    ds.attrs['ncfile']    = flare_exp_nc_file
    ds.attrs['delta_x']   = 1e3 * np.mean(np.diff(ds.longitude.values)) * 111.13295254925466
    ds.attrs['delta_y']   = 1e3 * np.mean(np.diff(ds.latitude.values)) * 111.13295254925466


    
    mod   = convert_units_3d(ds, ds["rho"])
    d_µm  = define_bin_boundaries() * 1.0e6 * 2.0 # radius to diameter, m to µm
    mod   = mod.assign_coords({'diameter_edges': xr.DataArray(d_µm, dims='diameter_edges')})
    
    if debug:
        print('DEBUGGING IS ON')
        mod = mod.isel(time=slice(None, None, debug_step))
        print('DEBUGGING IS ON')
        

    for tr_var in ['qi', 'qs']:
        tracks_file        =  f'{model_data_path}/{flare_exp_name}_{tr_var}_tobac_track.csv'
        features_mask_file =  f'{model_data_path}/{flare_exp_name}_{tr_var}_tobac_features_mask.csv'
        segm_mask_file     =  f'{model_data_path}/{flare_exp_name}_{tr_var}_tobac_mask_xarray.nc'
        
        tracks        = pd.read_csv(tracks_file).to_xarray()
        features_mask = pd.read_csv(features_mask_file).to_xarray()
        ds_segm       = xr.open_dataset(segm_mask_file)
        ds_segm       = ds_segm.sel(reduced_domain)
        ds_segm, cell_input  = xr.align(ds_segm, mod, join='inner', exclude=['altitude', 'latitude', 'longitude', 'diameter_edges'])
        
        delayed = []
        for type in cell_extraction_type:
            pl_ = extract_segmented_tracks(cell_input, tracks, ds_segm['segmentation_mask'], type=type)
            for key, val in pl_.items():
                # Avoid CF encoding conflict: drop any existing 'units' on time before writing
                if 'time' in val.coords:
                    val = val.copy()
                    val['time'].attrs.pop('units', None)
                    val['time'].attrs.pop('calendar', None)
                nc_f = f'processed/data_{cs_run}_{flare_exp_name}_{type}_plume_path_{tr_var}_cell{key}.nc'
                delayed.append(val.to_netcdf(nc_f, compute=False))
                print(f' delayed  {nc_f}')
            
        print('Finalizing delayed tasks')
        with ProgressBar():
            results = dask.compute(*delayed)

    print(f'    DONE: {cs_run}')


NameError: name 'cs_runs' is not defined